# South Barnegat Bay Onshore Wind Model Prediction with the Use of Long Short-Term Memory Neural Networks - LSTM Binary Model (Depreciated)

### Written by Krupam Patel
### Written in Python 3.11.14

###### This file contains the code for the Binary LSTM for upwelling and onshore flags prediction.

In [1]:
from tensorflow import keras # type: ignore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    classification_report,
    confusion_matrix
)
import os
import pandas as pd
import numpy as np

In [2]:
# Train one model at a time.
#TARGET = "Onshore"
TARGET = "upwelling_flag"

WINDOW = 24
THRESHOLD = 0.5

# Start without weighting
USE_CLASS_WEIGHTS = True

In [3]:
#Opens a new dataframe with the Clean csv
cleancsv = pd.read_csv('CLEAN.csv') 

In [4]:
cleancsv["Date"] = (
    pd.to_datetime(cleancsv["Date"], errors="coerce")
    + pd.to_timedelta(cleancsv["Hr"], unit="h")
)

cleancsv = cleancsv.drop(columns="Hr")
cleancsv = cleancsv.sort_values("Date").reset_index(drop=True)

if cleancsv["Date"].isna().any():
    raise ValueError("Fix invalid timestamps before training.")

if cleancsv["Date"].duplicated().any():
    raise ValueError("Resolve duplicate timestamps before training.")

# Use the same months across models for comparison.
summer_mask = cleancsv["Date"].dt.month.isin([6, 7, 8])
cleancsv = cleancsv.loc[summer_mask].reset_index(drop=True)

In [5]:
# Find all upelling EVENTS
event = cleancsv["upwelling_flag"].eq(1)

# Start a event after a non-event hour or time gap.
starts = event & (
    ~event.shift(1, fill_value=False)
    | cleancsv["Date"].diff().ne(pd.Timedelta(hours=1))
)

episode_id = starts.cumsum()

episodes = (
    cleancsv.loc[event]
    .groupby(episode_id.loc[event])
    .agg(
        start=("Date", "min"),
        end=("Date", "max"),
        labeled_hours=("Date", "size")
    )
)

print("Labeled upwelling episodes:")
print(episodes.to_string(index=False))

Labeled upwelling episodes:
              start                 end  labeled_hours
2025-06-02 17:00:00 2025-06-02 17:00:00              1
2025-06-06 16:00:00 2025-06-06 16:00:00              1
2025-06-11 15:00:00 2025-06-11 16:00:00              2
2025-06-13 07:00:00 2025-06-13 07:00:00              1
2025-06-13 09:00:00 2025-06-13 10:00:00              2
2025-06-19 14:00:00 2025-06-19 19:00:00              6
2025-06-21 16:00:00 2025-06-21 19:00:00              4
2025-06-24 18:00:00 2025-06-24 18:00:00              1
2025-06-28 16:00:00 2025-06-28 19:00:00              4
2025-07-03 17:00:00 2025-07-03 18:00:00              2
2025-07-03 21:00:00 2025-07-03 21:00:00              1
2025-07-05 15:00:00 2025-07-05 18:00:00              4
2025-07-09 19:00:00 2025-07-09 20:00:00              2
2025-07-15 18:00:00 2025-07-15 19:00:00              2
2025-07-29 18:00:00 2025-07-29 18:00:00              1
2025-07-30 17:00:00 2025-07-30 20:00:00              4
2025-08-11 19:00:00 2025-08-11 19:00:

In [6]:
# Validate the selected binary target BEFORE conversion.
target_values = cleancsv[TARGET].to_numpy()

if not np.isin(target_values, [0, 1]).all():
    raise ValueError(
        f"{TARGET} must contain only 0 and 1. "
        f"Found: {np.unique(target_values)}"
    )

output_data = target_values.astype(np.int32)

# Historical flags may be inputs: windows end before the target hour.
feature_cols = [
    column for column in cleancsv.columns
    if column != "Date"
]

dataset = cleancsv[feature_cols].to_numpy(dtype=np.float32)

if not np.isfinite(dataset).all():
    raise ValueError("Inputs contain missing or infinite values.")

In [7]:
# Length
n = len(cleancsv)
train_end = int(n * 0.80)
test_start = int(np.ceil(n * 0.90))

if not WINDOW < train_end < test_start < n:
    raise ValueError("Not enough observations for these splits.")

In [ ]:
scaler_x = StandardScaler()
scaler_x.fit(dataset[:train_end])
scaledx = scaler_x.transform(dataset)

# Require consecutive hours through the target timestamp.
hourly_step = (
    cleancsv["Date"].diff() == pd.Timedelta(hours=1)
)

valid_target = (
    hourly_step
    .rolling(window=WINDOW, min_periods=WINDOW)
    .sum()
    .eq(WINDOW)
)

target_rows = np.flatnonzero(valid_target.to_numpy())

if len(target_rows) == 0:
    raise ValueError("No complete 24-hour forecast windows found.")

In [9]:
X = np.stack([
    scaledx[i - WINDOW:i]
    for i in target_rows
])

# No target scaling or class-index subtraction.
y = output_data[target_rows].astype(np.float32).reshape(-1, 1)

train_mask = target_rows < train_end
val_mask = (
    (target_rows >= train_end)
    & (target_rows < test_start)
)
test_mask = target_rows >= test_start

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

test_rows = target_rows[test_mask]

# Inspect class availability in each period.
for name, labels in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Testing", y_test)
]:
    if labels.size == 0:
        raise ValueError(f"{name} split has no valid windows.")

    counts = np.bincount(
        labels.ravel().astype(int),
        minlength=2
    )

    print(f"{name}: zeros={counts[0]}, ones={counts[1]}")

    if np.any(counts == 0):
        print(f"WARNING: {name} contains only one class.")

if np.unique(y_train).size < 2:
    raise ValueError(
        f"Training data for {TARGET} contain only one class. "
        "More data containing both 0 and 1 are needed."
    )

print("Training shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Testing shape:", X_test.shape, y_test.shape)

Training: zeros=1685, ones=38
Validation: zeros=219, ones=0
Testing: zeros=218, ones=0
Training shape: (1723, 24, 13) (1723, 1)
Validation shape: (219, 24, 13) (219, 1)
Testing shape: (218, 24, 13) (218, 1)


In [10]:
class_weight = None

if USE_CLASS_WEIGHTS:
    counts = np.bincount(
        y_train.ravel().astype(int),
        minlength=2
    )

    class_weight = {
        0: float(len(y_train) / (2 * counts[0])),
        1: float(len(y_train) / (2 * counts[1]))
    }

    print("Class weights:", class_weight)

Class weights: {0: 0.5112759643916914, 1: 22.67105263157895}


In [11]:
print("X_train:", X_train.shape)   # (samples, 24, 13)
print("y_train:", y_train.shape)   # (samples, 1)
print("First 10 labels:", y_train[:10].ravel())

X_train: (1723, 24, 13)
y_train: (1723, 1)
First 10 labels: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [12]:
keras.backend.clear_session()
keras.utils.set_random_seed(42)

model = keras.models.Sequential([
    keras.layers.Input(shape=(WINDOW, len(feature_cols))),
    keras.layers.LSTM(32),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

In [13]:
#Put all the layers together
model.summary()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         5,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,977 (27.25 KB)

 Trainable params: 6,977 (27.25 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
#Train the model

#epochs = # of runs
#batch size = how much data is in each batch
training = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    shuffle=False,
    class_weight=class_weight,
    callbacks=[early_stopping]
)

Epoch 1/200
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8149 - loss: 0.6640 - precision: 0.0423 - recall: 0.3421 - val_accuracy: 0.9635 - val_loss: 0.4110 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/200
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8369 - loss: 0.5114 - precision: 0.1016 - recall: 0.8158 - val_accuracy: 0.9635 - val_loss: 0.2639 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/200
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8305 - loss: 0.3958 - precision: 0.1056 - recall: 0.8947 - val_accuracy: 0.9726 - val_loss: 0.1761 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/200
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8334 - loss: 0.3128 - precision: 0.1121 - recall: 0.9474 - val_accuracy: 0.9817 - val_loss: 0.1166 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/200
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8555 - loss: 0.2578 - precision: 0.1298 - recall: 0.9737 - val_accu

In [15]:
for name, inputs, labels in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val)
]:
    actual = np.asarray(labels).ravel().astype(int)
    probabilities = model.predict(inputs, verbose=0).ravel()
    flags = (probabilities >= 0.5).astype(int)

    print(f"\n{name}")
    print(classification_report(
        actual,
        flags,
        labels=[0, 1],
        target_names=["No event", "Event"],
        zero_division=0
    ))

    for label in [0, 1]:
        mask = actual == label

        if mask.any():
            print(f"Mean predicted probability for actual {label}s: {probabilities[mask].mean():.6f}")


Training
              precision    recall  f1-score   support

    No event       1.00      0.97      0.98      1685
       Event       0.43      1.00      0.60        38

    accuracy                           0.97      1723
   macro avg       0.71      0.98      0.79      1723
weighted avg       0.99      0.97      0.98      1723

Mean predicted probability for actual 0s: 0.041306
Mean predicted probability for actual 1s: 0.965309

Validation
              precision    recall  f1-score   support

    No event       1.00      1.00      1.00       219
       Event       0.00      0.00      0.00         0

    accuracy                           1.00       219
   macro avg       0.50      0.50      0.50       219
weighted avg       1.00      1.00      1.00       219

Mean predicted probability for actual 0s: 0.001414


In [16]:
prediction_prob = model.predict(X_test).ravel()

prediction_flag = (
    prediction_prob >= THRESHOLD
).astype(np.int32)

y_test_true = y_test.ravel().astype(np.int32)

# Persistence: next hour has the previous hour's flag.
naive_prediction = output_data[test_rows - 1]

# Additional baseline: always predict no event.
zero_prediction = np.zeros_like(y_test_true)

print(f"\nTarget: {TARGET}")
print(f"Threshold: {THRESHOLD}")

for name, predicted in [
    ("LSTM", prediction_flag),
    ("Persistence", naive_prediction),
    ("Always zero", zero_prediction)
]:
    mae = mean_absolute_error(y_test_true, predicted)

    print(f"\n{name} flag MAE: {mae:.4f}")
    print(f"{name} incorrect forecasts: {mae * 100:.2f}%")

    print(classification_report(
        y_test_true,
        predicted,
        labels=[0, 1],
        target_names=["No event", "Event"],
        zero_division=0
    ))

print("LSTM confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(
    y_test_true,
    prediction_flag,
    labels=[0, 1]
))

if np.unique(y_test_true).size < 2:
    print("WARNING: The test set contains only one class. cannot assess performance on both event and non-event hours.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 

Target: upwelling_flag
Threshold: 0.5

LSTM flag MAE: 0.0229
LSTM incorrect forecasts: 2.29%
              precision    recall  f1-score   support

    No event       1.00      0.98      0.99       218
       Event       0.00      0.00      0.00         0

    accuracy                           0.98       218
   macro avg       0.50      0.49      0.49       218
weighted avg       1.00      0.98      0.99       218


Persistence flag MAE: 0.0000
Persistence incorrect forecasts: 0.00%
              precision    recall  f1-score   support

    No event       1.00      1.00      1.00       218
       Event       0.00      0.00      0.00         0

    accuracy                           1.00       218
   macro avg       0.50      0.50      0.50       218
weighted avg       1.00      1.00      1.00       218


Always zero flag MAE: 0.0000
Always zero incorrect forecasts: 0.00%
              precision    recall  f1-score   support

    No event       1.

In [17]:
test_df = pd.DataFrame({
    "Date": cleancsv["Date"].iloc[test_rows].to_numpy(),
    f"observed_{TARGET}": y_test_true,
    f"predicted_{TARGET}_prob": prediction_prob,
    f"predicted_{TARGET}_flag": prediction_flag,
    f"predicted_{TARGET}_naive": naive_prediction
})

os.makedirs("CSV/predictions", exist_ok=True)

output_path = f"CSV/predictions/{TARGET}_lstm_predictions.csv"

test_df.to_csv(output_path, index=False)
print("Saved:", output_path)

Saved: CSV/predictions/upwelling_flag_lstm_predictions.csv
